In [1]:
!pip install torch_geometric 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 8.3 MB/s  0:00:00


In [2]:

# ================================================================
# KNN SENSITIVITY EXPERIMENT: GATv2 SPATIAL ENCODER
# K = 4, 8, 12 | Seeds = 42, 123, 2023
# ================================================================

import os
import gc
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from pathlib import Path
from sklearn.neighbors import BallTree
from torch_geometric.data import Data
from torch_geometric.nn import GATv2Conv


# ================================================================
# 1. CONFIGURATION
# ================================================================

DATA_DIR = Path(
    "/kaggle/input/datasets/drrajgauravmishra/"
    "spatiotemporal-gnn-kaggle-data/"
    "Spatiotemporal_GNN_Kaggle_Data"
)

NODES_PATH = Path(
    "/kaggle/input/datasets/drrajgauravmishra/"
    "spatial-nodes/spatial_nodes.csv"
)

OUTPUT_DIR = Path("/kaggle/working/spatial_knn_experiments")
GRAPH_DIR = OUTPUT_DIR / "graphs"
MODEL_DIR = OUTPUT_DIR / "models"
HISTORY_DIR = OUTPUT_DIR / "histories"

for directory in [OUTPUT_DIR, GRAPH_DIR, MODEL_DIR, HISTORY_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

RESULTS_FILE = OUTPUT_DIR / "spatial_knn_sensitivity_results.csv"

K_VALUES = [4, 8, 12]
SEEDS = [42, 123, 2023]

EPOCHS = 1000
LEARNING_RATE = 0.0005
HIDDEN_DIM = 64

DEVICE = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

print("=" * 70)
print("SPATIAL KNN SENSITIVITY EXPERIMENT")
print("=" * 70)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("PyTorch:", torch.__version__)


# ================================================================
# 2. LOAD AND RECONSTRUCT ANNUAL SNAPSHOTS
# ================================================================

print("\n[1/5] Loading annual datasets...")

datasets = {
    "training": np.load(DATA_DIR / "training_annual.npz"),
    "validation": np.load(DATA_DIR / "validation_annual.npz"),
    "testing": np.load(DATA_DIR / "testing_annual.npz")
}

snapshot_dict = {}

for split_name, dataset in datasets.items():

    X = dataset["X"]
    input_indices = dataset["input_indices"]

    for sample_idx in range(X.shape[0]):

        for time_idx, year_idx in enumerate(
            input_indices[sample_idx]
        ):

            year_idx = int(year_idx)
            snapshot = X[sample_idx, time_idx]

            if year_idx in snapshot_dict:

                if not np.allclose(
                    snapshot_dict[year_idx],
                    snapshot,
                    rtol=1e-5,
                    atol=1e-6,
                    equal_nan=True
                ):
                    raise ValueError(
                        f"Inconsistent snapshot at index {year_idx}"
                    )

            else:
                snapshot_dict[year_idx] = snapshot.copy()


expected_indices = list(range(25))

if sorted(snapshot_dict.keys()) != expected_indices:
    raise ValueError(
        "Annual snapshot reconstruction failed. "
        f"Found indices: {sorted(snapshot_dict.keys())}"
    )

annual_snapshots = np.stack(
    [snapshot_dict[i] for i in expected_indices],
    axis=0
)

print("Reconstructed snapshots:", annual_snapshots.shape)

assert annual_snapshots.shape == (25, 5040, 4)


# ================================================================
# 3. PREPARE FEATURES AND OBSERVATION MASKS
# ================================================================

print("\n[2/5] Preparing spatial features and masks...")

spatial_x = torch.tensor(
    annual_snapshots[..., :2],
    dtype=torch.float32,
    device=DEVICE
)

missing_indicators = annual_snapshots[..., 2:4]

observed_mask = torch.tensor(
    missing_indicators != 1.0,
    dtype=torch.bool,
    device=DEVICE
)

# Spatial encoder training: 2000-2023
train_x = spatial_x[:24]
train_mask = observed_mask[:24]

# Reserved for embedding extraction: 2024
future_x = spatial_x[24]
future_mask = observed_mask[24]

print("Training features:", tuple(train_x.shape))
print("Training mask:", tuple(train_mask.shape))
print("Reserved 2024 snapshot:", tuple(future_x.shape))
print("Observed training values:", train_mask.sum().item())


# ================================================================
# 4. LOAD AND GENERATE SPATIAL GRAPHS
# ================================================================

print("\n[3/5] Preparing KNN graphs...")

# Load spatial coordinates
spatial_nodes = pd.read_csv(NODES_PATH)

if not {"latitude", "longitude"}.issubset(
    spatial_nodes.columns
):
    raise ValueError(
        "The spatial coordinates file must contain "
        "'latitude' and 'longitude' columns."
    )

if len(spatial_nodes) != 5040:
    raise ValueError(
        f"Expected 5040 spatial nodes, found {len(spatial_nodes)}"
    )

coords = np.radians(
    spatial_nodes[["latitude", "longitude"]].to_numpy()
)

# Haversine nearest-neighbour search
tree = BallTree(coords, metric="haversine")

edge_indices = {}

# Generate K=4 and K=12 graphs
for k in [4, 12]:

    graph_path = GRAPH_DIR / f"knn_graph_k{k}.pt"

    distances, indices = tree.query(
        coords,
        k=k + 1
    )

    edge_sources = np.repeat(
        np.arange(len(coords)),
        k
    )

    edge_targets = indices[:, 1:].reshape(-1)

    edge_index = torch.tensor(
        np.stack([edge_sources, edge_targets]),
        dtype=torch.long
    )

    graph_data = Data(
        edge_index=edge_index,
        num_nodes=len(coords)
    )

    torch.save(graph_data, graph_path)

    print(
        f"K={k}: {graph_data.num_nodes} nodes, "
        f"{graph_data.edge_index.shape[1]} edges"
    )


# Load original K=8 graph
original_graph_path = (
    DATA_DIR / "static_spatial_graph_verified.pt"
)

graph_k8 = torch.load(
    original_graph_path,
    map_location="cpu",
    weights_only=False
)

if graph_k8.num_nodes != 5040:
    raise ValueError("Unexpected node count in original K=8 graph.")

edge_indices[8] = graph_k8.edge_index.to(DEVICE)

# Load generated K=4 and K=12 graphs
for k in [4, 12]:

    graph_path = GRAPH_DIR / f"knn_graph_k{k}.pt"

    graph_data = torch.load(
        graph_path,
        map_location="cpu",
        weights_only=False
    )

    edge_indices[k] = graph_data.edge_index.to(DEVICE)


# Verify graph structures
for k in K_VALUES:

    edge_index = edge_indices[k]

    if edge_index.shape[1] != 5040 * k:
        raise ValueError(
            f"Unexpected edge count for K={k}: "
            f"{edge_index.shape[1]}"
        )

    neighbour_counts = torch.bincount(
        edge_index[0].cpu(),
        minlength=5040
    )

    if not torch.all(neighbour_counts == k):
        raise ValueError(
            f"Incorrect neighbour count for K={k}"
        )

    if torch.any(edge_index[0] == edge_index[1]):
        raise ValueError(
            f"Self-loops detected in K={k} graph"
        )

    print(
        f"Verified K={k}: "
        f"{edge_index.shape[1]} directed edges"
    )


# ================================================================
# 5. DEFINE GATv2 ARCHITECTURE
# ================================================================

class SpatialGATv2(nn.Module):

    def __init__(self, in_dim=2, hidden_dim=64):

        super().__init__()

        self.gat1 = GATv2Conv(
            in_dim,
            hidden_dim,
            heads=4,
            concat=True,
            add_self_loops=False
        )

        self.gat2 = GATv2Conv(
            hidden_dim * 4,
            2,
            heads=1,
            concat=True,
            add_self_loops=False
        )

    def forward(self, x, edge_index):

        x = F.relu(
            self.gat1(x, edge_index)
        )

        x = self.gat2(x, edge_index)

        return x


def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def masked_reconstruction_mse(
    prediction,
    target,
    mask
):

    return F.mse_loss(
        prediction[mask],
        target[mask]
    )


print("\nGATv2 architecture defined.")

model_check = SpatialGATv2()

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in model_check.parameters()
        if p.requires_grad
    )
)

del model_check
gc.collect()


# ================================================================
# 6. RESUMABLE TRAINING FOR ALL NINE CONFIGURATIONS
# ================================================================

print("\n[4/5] Starting KNN sensitivity experiments...")

if RESULTS_FILE.exists():

    results_df = pd.read_csv(RESULTS_FILE)

else:

    results_df = pd.DataFrame(
        columns=[
            "K",
            "Seed",
            "Final_MSE",
            "Training_Time_Min"
        ]
    )


def save_results(records):

    df = pd.DataFrame(records)

    df = df.sort_values(
        ["K", "Seed"]
    ).reset_index(drop=True)

    df.to_csv(
        RESULTS_FILE,
        index=False
    )

    return df


for k in K_VALUES:

    for seed in SEEDS:

        model_path = (
            MODEL_DIR / f"gatv2_k{k}_seed{seed}.pt"
        )

        history_path = (
            HISTORY_DIR / f"history_k{k}_seed{seed}.npy"
        )

        # Resume: skip a previously completed configuration
        if model_path.exists() and history_path.exists():

            try:

                checkpoint = torch.load(
                    model_path,
                    map_location="cpu",
                    weights_only=False
                )

                history = np.load(history_path)

                if (
                    len(history) == EPOCHS
                    and checkpoint["K"] == k
                    and checkpoint["seed"] == seed
                ):

                    print(
                        f"\nSkipping completed "
                        f"K={k}, Seed={seed}"
                    )

                    existing = results_df[
                        (results_df["K"] == k)
                        & (results_df["Seed"] == seed)
                    ]

                    if existing.empty:

                        results_df = pd.concat(
                            [
                                results_df,
                                pd.DataFrame([{
                                    "K": k,
                                    "Seed": seed,
                                    "Final_MSE": float(history[-1]),
                                    "Training_Time_Min":
                                        checkpoint.get(
                                            "training_time_min",
                                            np.nan
                                        )
                                }])
                            ],
                            ignore_index=True
                        )

                        results_df = save_results(
                            results_df.to_dict("records")
                        )

                    continue

            except Exception as error:

                print(
                    "Existing checkpoint could not be reused:",
                    error
                )


        print("\n" + "=" * 65)
        print(f"TRAINING GATv2 | K={k} | SEED={seed}")
        print("=" * 65)

        set_seed(seed)

        model = SpatialGATv2().to(DEVICE)

        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=LEARNING_RATE
        )

        edge_index = edge_indices[k]

        history = []

        start_time = time.time()

        for epoch in range(1, EPOCHS + 1):

            model.train()
            optimizer.zero_grad()

            year_losses = []

            for year_idx in range(24):

                reconstruction = model(
                    train_x[year_idx],
                    edge_index
                )

                loss_year = masked_reconstruction_mse(
                    reconstruction,
                    train_x[year_idx],
                    train_mask[year_idx]
                )

                year_losses.append(loss_year)

            loss = torch.stack(year_losses).mean()

            loss.backward()
            optimizer.step()

            epoch_loss = loss.item()

            history.append(epoch_loss)

            if epoch == 1 or epoch % 100 == 0:

                print(
                    f"K={k} | Seed={seed} | "
                    f"Epoch={epoch}/{EPOCHS} | "
                    f"MSE={epoch_loss:.6f}"
                )


        elapsed = time.time() - start_time
        elapsed_min = elapsed / 60

        # Save model checkpoint
        checkpoint = {
            "model_state_dict": model.state_dict(),
            "K": k,
            "seed": seed,
            "epochs": EPOCHS,
            "learning_rate": LEARNING_RATE,
            "hidden_dim": HIDDEN_DIM,
            "final_mse": history[-1],
            "training_time_min": elapsed_min
        }

        temporary_model_path = Path(
            str(model_path) + ".tmp"
        )

        torch.save(
            checkpoint,
            temporary_model_path
        )

        os.replace(
            temporary_model_path,
            model_path
        )

        # Save training history
        np.save(
            history_path,
            np.asarray(history, dtype=np.float32)
        )

        # Update results immediately
        new_record = {
            "K": k,
            "Seed": seed,
            "Final_MSE": history[-1],
            "Training_Time_Min": elapsed_min
        }

        results_df = pd.concat(
            [
                results_df,
                pd.DataFrame([new_record])
            ],
            ignore_index=True
        )

        results_df = results_df.drop_duplicates(
            subset=["K", "Seed"],
            keep="last"
        )

        results_df = save_results(
            results_df.to_dict("records")
        )

        print("\nConfiguration completed.")
        print(f"Final MSE: {history[-1]:.6f}")
        print(f"Training time: {elapsed_min:.2f} minutes")
        print("Checkpoint:", model_path)

        # Release memory
        del model
        del optimizer
        del history
        del year_losses
        del loss

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# ================================================================
# 7. FINAL RESULTS
# ================================================================

print("\n[5/5] EXPERIMENT SUMMARY")

results_df = pd.read_csv(RESULTS_FILE)

results_df = results_df.sort_values(
    ["K", "Seed"]
).reset_index(drop=True)

print("\nIndividual experiment results:")
display(results_df)

print("\nMean and standard deviation by K:")

summary_df = (
    results_df.groupby("K")["Final_MSE"]
    .agg(["mean", "std"])
    .reset_index()
)

display(summary_df)

summary_df.to_csv(
    OUTPUT_DIR / "spatial_knn_sensitivity_summary.csv",
    index=False
)

print("\n" + "=" * 70)
print("KNN SENSITIVITY EXPERIMENT FINISHED")
print("=" * 70)

print("Completed configurations:", len(results_df))
print("Expected configurations:", len(K_VALUES) * len(SEEDS))
print("Output directory:", OUTPUT_DIR)
print("Results CSV:", RESULTS_FILE)
print("Summary CSV:", OUTPUT_DIR / "spatial_knn_sensitivity_summary.csv")
print("Model checkpoints:", MODEL_DIR)
print("Training histories:", HISTORY_DIR)

if len(results_df) != 9:
    print("\nWARNING: Not all nine configurations are complete.")
else:
    print("\nAll nine configurations completed successfully.")


SPATIAL KNN SENSITIVITY EXPERIMENT
Device: cuda:0
GPU: Tesla T4
PyTorch: 2.11.0+cu130

[1/5] Loading annual datasets...
Reconstructed snapshots: (25, 5040, 4)

[2/5] Preparing spatial features and masks...
Training features: (24, 5040, 2)
Training mask: (24, 5040, 2)
Reserved 2024 snapshot: (5040, 2)
Observed training values: 220138

[3/5] Preparing KNN graphs...
K=4: 5040 nodes, 20160 edges
K=12: 5040 nodes, 60480 edges
Verified K=4: 20160 directed edges
Verified K=8: 40320 directed edges
Verified K=12: 60480 directed edges

GATv2 architecture defined.
Trainable parameters: 3080

[4/5] Starting KNN sensitivity experiments...

TRAINING GATv2 | K=4 | SEED=42
K=4 | Seed=42 | Epoch=1/1000 | MSE=1.803669
K=4 | Seed=42 | Epoch=100/1000 | MSE=0.115984
K=4 | Seed=42 | Epoch=200/1000 | MSE=0.033501
K=4 | Seed=42 | Epoch=300/1000 | MSE=0.029912
K=4 | Seed=42 | Epoch=400/1000 | MSE=0.028218
K=4 | Seed=42 | Epoch=500/1000 | MSE=0.026765
K=4 | Seed=42 | Epoch=600/1000 | MSE=0.025376
K=4 | Seed=42 

/tmp/ipykernel_24/445421622.py:584: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_df = pd.concat(



TRAINING GATv2 | K=4 | SEED=123
K=4 | Seed=123 | Epoch=1/1000 | MSE=1.047975
K=4 | Seed=123 | Epoch=100/1000 | MSE=0.044809
K=4 | Seed=123 | Epoch=200/1000 | MSE=0.022808
K=4 | Seed=123 | Epoch=300/1000 | MSE=0.020816
K=4 | Seed=123 | Epoch=400/1000 | MSE=0.019520
K=4 | Seed=123 | Epoch=500/1000 | MSE=0.018583
K=4 | Seed=123 | Epoch=600/1000 | MSE=0.017903
K=4 | Seed=123 | Epoch=700/1000 | MSE=0.017406
K=4 | Seed=123 | Epoch=800/1000 | MSE=0.017008
K=4 | Seed=123 | Epoch=900/1000 | MSE=0.016665
K=4 | Seed=123 | Epoch=1000/1000 | MSE=0.016334

Configuration completed.
Final MSE: 0.016334
Training time: 2.21 minutes
Checkpoint: /kaggle/working/spatial_knn_experiments/models/gatv2_k4_seed123.pt

TRAINING GATv2 | K=4 | SEED=2023
K=4 | Seed=2023 | Epoch=1/1000 | MSE=1.195641
K=4 | Seed=2023 | Epoch=100/1000 | MSE=0.059374
K=4 | Seed=2023 | Epoch=200/1000 | MSE=0.031760
K=4 | Seed=2023 | Epoch=300/1000 | MSE=0.027657
K=4 | Seed=2023 | Epoch=400/1000 | MSE=0.025510
K=4 | Seed=2023 | Epoch=50

,K,Seed,Final_MSE,Training_Time_Min
0,4,42,0.021749,2.228189
1,4,123,0.016334,2.213512
2,4,2023,0.021736,2.223454
3,8,42,0.036363,3.500174
4,8,123,0.031220,3.497955
5,8,2023,0.031234,3.504864
6,12,42,0.039972,4.806990
7,12,123,0.043750,4.803699
8,12,2023,0.040004,4.825120



Mean and standard deviation by K:


,K,mean,std
0,4,0.019940,0.003123
1,8,0.032939,0.002965
2,12,0.041242,0.002172



KNN SENSITIVITY EXPERIMENT FINISHED
Completed configurations: 9
Expected configurations: 9
Output directory: /kaggle/working/spatial_knn_experiments
Results CSV: /kaggle/working/spatial_knn_experiments/spatial_knn_sensitivity_results.csv
Summary CSV: /kaggle/working/spatial_knn_experiments/spatial_knn_sensitivity_summary.csv
Model checkpoints: /kaggle/working/spatial_knn_experiments/models
Training histories: /kaggle/working/spatial_knn_experiments/histories

All nine configurations completed successfully.
